# DSA210 Milestone 1  
## Movie Success Analysis Using TMDB and IMDb Data

This notebook covers the April 14 milestone requirements:
- data collection
- data cleaning and merging
- exploratory data analysis (EDA)
- hypothesis testing


## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

plt.rcParams["figure.figsize"] = (8, 5)

## 2. Load Data

Place these files in the same folder as this notebook:
- `tmdb_5000_movies.csv`
- `title.basics.tsv`
- `title.ratings.tsv`

If your files are in another folder, replace the filenames below with full paths.


In [ ]:
tmdb = pd.read_csv("tmdb_5000_movies.csv")
basics = pd.read_csv("title.basics.tsv", sep="\t")
ratings = pd.read_csv("title.ratings.tsv", sep="\t")

print("TMDB shape:", tmdb.shape)
print("IMDb basics shape:", basics.shape)
print("IMDb ratings shape:", ratings.shape)

## 3. Clean IMDb Data

We keep only movies and only the columns needed for this project.


In [ ]:
basics = basics[basics["titleType"] == "movie"]
basics = basics[["tconst", "primaryTitle", "startYear"]]
ratings = ratings[["tconst", "averageRating", "numVotes"]]

imdb = pd.merge(basics, ratings, on="tconst")
imdb.rename(columns={"primaryTitle": "title", "startYear": "year"}, inplace=True)

print("IMDb merged shape:", imdb.shape)
imdb.head()

## 4. Clean TMDB Data

We keep the main movie-related columns and remove entries with zero budget or zero revenue so that financial analysis is meaningful.


In [ ]:
tmdb = tmdb[["title", "budget", "revenue", "popularity", "genres", "release_date"]]
tmdb = tmdb[(tmdb["budget"] > 0) & (tmdb["revenue"] > 0)]

print("TMDB cleaned shape:", tmdb.shape)
tmdb.head()

## 5. Merge TMDB and IMDb

The two datasets are merged on movie title to combine financial data with audience-based data.


In [ ]:
df = pd.merge(tmdb, imdb, on="title")

print("Final merged shape:", df.shape)
df.head()

## 6. Feature Engineering / Enrichment

New variables are created to improve the analysis:
- `profit = revenue - budget`
- `ROI = revenue / budget`
- release decade
- success category based on profit


**Interpretation:**  
The number of movies increases in recent decades. This may reflect growth in the film industry or dataset bias toward more recent films.

In [ ]:
df["profit"] = df["revenue"] - df["budget"]
df["ROI"] = df["revenue"] / df["budget"]

df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["year_tmdb"] = df["release_date"].dt.year
df["decade"] = (df["year_tmdb"] // 10) * 10

def success_category(profit):
    if profit < 0:
        return "Low"
    elif profit < 100_000_000:
        return "Medium"
    else:
        return "High"

df["success_category"] = df["profit"].apply(success_category)

df[["title", "budget", "revenue", "averageRating", "numVotes", "profit", "ROI", "decade", "success_category"]].head()

## 7. Exploratory Data Analysis (EDA)

### 7.1 Budget vs Revenue
This scatter plot shows whether higher-budget films tend to generate higher revenue.


**Interpretation:**  
There is a clear positive relationship between budget and revenue. Movies with higher budgets tend to generate higher revenue, although the spread is large. This suggests that budget is an important factor but not the only determinant of success.

In [ ]:
plt.scatter(df["budget"], df["revenue"])
plt.xlabel("Budget")
plt.ylabel("Revenue")
plt.title("Budget vs Revenue")
plt.show()

**Interpretation:**  
The scatter plot suggests a positive relationship between budget and revenue. Movies with larger budgets tend to generate higher revenues on average, although the spread is quite wide. This means budget appears to be an important factor, but it is not sufficient on its own to explain movie success.


### 7.2 Rating vs Revenue
This scatter plot shows the relationship between IMDb rating and revenue.


**Interpretation:**  
The relationship between IMDb rating and revenue is weak. Although some highly rated movies earn high revenue, many do not. This indicates that rating alone is not a strong predictor of financial success.

In [ ]:
plt.scatter(df["averageRating"], df["revenue"])
plt.xlabel("IMDb Rating")
plt.ylabel("Revenue")
plt.title("Rating vs Revenue")
plt.show()

**Interpretation:**  
The relationship between IMDb rating and revenue appears weaker than the relationship between budget and revenue. While some highly rated movies earn very large revenue, many highly rated films still have moderate or low revenue. This suggests that audience rating may matter, but financial performance is influenced by several other factors.


### 7.3 Profit Distribution
This histogram shows how movie profits are distributed.


**Interpretation:**  
The distribution is heavily right-skewed. Most movies have relatively low profit, while a small number achieve very high profits. This shows that success is uneven and dominated by a few blockbuster films.

In [ ]:
df["profit"].hist()
plt.title("Profit Distribution")
plt.xlabel("Profit")
plt.ylabel("Frequency")
plt.show()

**Interpretation:**  
The profit distribution is strongly right-skewed. Most movies have relatively modest profit values, while a small number of films achieve extremely large profits. This suggests that movie profitability is unevenly distributed and includes several strong outliers.


**Interpretation:**  
The distribution is heavily right-skewed. Most movies have relatively low profit, while a small number achieve very high profits. This shows that success is uneven and dominated by a few blockbuster films.

### 7.4 Number of Movies by Decade
This plot helps us observe whether the number of movies in the dataset changes across decades.


**Interpretation:**  
The number of movies increases in recent decades. This may reflect growth in the film industry or dataset bias toward more recent films.

In [ ]:
df["decade"].value_counts().sort_index().plot(kind="bar")
plt.title("Number of Movies by Decade")
plt.xlabel("Decade")
plt.ylabel("Count")
plt.show()

**Interpretation:**  
This bar chart shows how movies are distributed across release decades in the merged dataset. It helps us understand the temporal structure of the data and whether some decades are represented more heavily than others.


**Interpretation:**  
The number of movies increases in recent decades. This may reflect growth in the film industry or dataset bias toward more recent films.

### 7.5 Budget vs Revenue (Log Scale)
Because budget and revenue contain very large outliers, a log-scale plot can make the relationship easier to see.


**Interpretation:**  
There is a clear positive relationship between budget and revenue. Movies with higher budgets tend to generate higher revenue, although the spread is large. This suggests that budget is an important factor but not the only determinant of success.

**Interpretation:**  
Using a log scale makes the relationship between budget and revenue clearer by reducing the effect of extreme values. The positive trend remains visible.

In [ ]:
plt.scatter(df["budget"], df["revenue"])
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Budget (log scale)")
plt.ylabel("Revenue (log scale)")
plt.title("Budget vs Revenue (Log Scale)")
plt.show()

**Interpretation:**  
The log-scale version makes the positive relationship between budget and revenue easier to observe by compressing extreme values. This confirms the general upward trend while making the structure of the data clearer.


## 8. Hypothesis Testing

### Hypothesis Test 1: Budget and Revenue

- **H0:** There is no linear relationship between budget and revenue.  
- **H1:** There is a linear relationship between budget and revenue.


In [ ]:
corr_budget_revenue, p_budget_revenue = pearsonr(df["budget"], df["revenue"])
print("Budget-Revenue correlation:", corr_budget_revenue)
print("p-value:", p_budget_revenue)

**Interpretation:**  
If the p-value is smaller than 0.05, we reject H0 and conclude that budget and revenue have a statistically significant linear relationship.


### Hypothesis Test 2: IMDb Rating and Revenue

- **H0:** There is no linear relationship between IMDb rating and revenue.  
- **H1:** There is a linear relationship between IMDb rating and revenue.


In [ ]:
corr_rating_revenue, p_rating_revenue = pearsonr(df["averageRating"], df["revenue"])
print("Rating-Revenue correlation:", corr_rating_revenue)
print("p-value:", p_rating_revenue)

**Interpretation:**  
If the p-value is smaller than 0.05, we reject H0 and conclude that IMDb rating and revenue have a statistically significant linear relationship.


## 9. Short Conclusion

This milestone combined financial movie data from TMDB with audience-based data from IMDb. After cleaning and merging the datasets, exploratory analysis showed a positive relationship between budget and revenue, while the relationship between IMDb rating and revenue appeared weaker. Profit values were highly skewed, with a small number of movies earning very large profits. Hypothesis tests were included to evaluate the statistical significance of the observed relationships.


## 10. Optional: Save Cleaned Dataset
You can run the following line to save the merged and cleaned dataset for later milestones.


In [ ]:
df.to_csv("cleaned_movies.csv", index=False)

## Hypothesis Interpretations

**Budget vs Revenue:**  
There is a strong positive correlation (~0.68) with a very small p-value, so we reject H0 and conclude that budget significantly affects revenue.

**Rating vs Revenue:**  
The correlation is very weak (~0.087), although the p-value is small. This means the relationship is statistically significant but not practically meaningful.

# Machine Learning Models

In this section, machine learning methods are applied to the cleaned and merged movie dataset.  
The main goal is to predict movie revenue using variables such as budget, popularity, IMDb rating, and number of votes.

This part corresponds to the May 5 milestone: applying ML methods on the dataset.


## 1. Preparing Data for Machine Learning

For this regression task, the target variable is `revenue`.  
The selected input features are:

- `budget`
- `popularity`
- `averageRating`
- `numVotes`

These variables are chosen because they represent both financial/metadata information from TMDB and audience-based information from IMDb.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Select features and target
features = ["budget", "popularity", "averageRating", "numVotes"]
target = "revenue"

# Remove rows with missing values in selected columns
ml_df = df[features + [target]].dropna()

X = ml_df[features]
y = ml_df[target]

print("ML dataset shape:", ml_df.shape)
X.head()

## 2. Train-Test Split

The dataset is split into training and testing sets.  
The model learns from the training set and is evaluated on unseen test data.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

## 3. Baseline Model: Linear Regression

Linear Regression is used as a simple baseline model.  
It assumes a linear relationship between the input variables and movie revenue.


In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)

## 4. Decision Tree Regressor

A Decision Tree Regressor is also applied because movie revenue may depend on nonlinear relationships between variables.


In [ ]:
tree_model = DecisionTreeRegressor(random_state=42, max_depth=5)
tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)

## 5. Random Forest Regressor

A Random Forest Regressor is used as an additional model.  
It combines multiple decision trees and can often perform better than a single tree by reducing overfitting.


In [ ]:
forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    max_depth=8
)

forest_model.fit(X_train, y_train)

forest_pred = forest_model.predict(X_test)

## 6. Model Evaluation

The models are evaluated using:

- **MAE**: Mean Absolute Error
- **RMSE**: Root Mean Squared Error
- **R² Score**: How much variance in revenue is explained by the model

Lower MAE/RMSE is better.  
Higher R² is better.


In [ ]:
def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    }

results = [
    evaluate_model("Linear Regression", y_test, linear_pred),
    evaluate_model("Decision Tree", y_test, tree_pred),
    evaluate_model("Random Forest", y_test, forest_pred)
]

results_df = pd.DataFrame(results)
results_df

**Interpretation:**  
The model with the lowest RMSE and highest R² score performs best on the test set.  
Linear Regression provides a simple baseline, while Decision Tree and Random Forest can capture more complex relationships.  
If Random Forest performs best, this suggests that movie revenue is affected by nonlinear interactions between features.


## 7. Actual vs Predicted Revenue

The following plots compare actual revenue values with predicted revenue values.  
A better model should have points closer to the diagonal pattern.


In [ ]:
plt.scatter(y_test, linear_pred)
plt.xlabel("Actual Revenue")
plt.ylabel("Predicted Revenue")
plt.title("Linear Regression: Actual vs Predicted Revenue")
plt.show()

In [ ]:
plt.scatter(y_test, tree_pred)
plt.xlabel("Actual Revenue")
plt.ylabel("Predicted Revenue")
plt.title("Decision Tree: Actual vs Predicted Revenue")
plt.show()

In [ ]:
plt.scatter(y_test, forest_pred)
plt.xlabel("Actual Revenue")
plt.ylabel("Predicted Revenue")
plt.title("Random Forest: Actual vs Predicted Revenue")
plt.show()

**Interpretation:**  
The actual vs predicted plots help visualize how close each model's predictions are to real revenue values.  
Large deviations indicate prediction errors, which are expected because movie revenue is influenced by many factors that are not fully captured in this dataset.


## 8. Feature Importance

For the Random Forest model, feature importance is used to see which variables contribute most to revenue prediction.


In [ ]:
feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": forest_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance

In [ ]:
plt.bar(feature_importance["Feature"], feature_importance["Importance"])
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.title("Random Forest Feature Importance")
plt.xticks(rotation=45)
plt.show()

**Interpretation:**  
Feature importance shows which variables are most useful for predicting revenue.  
If budget has the highest importance, this supports the earlier EDA and hypothesis testing results showing that budget is strongly related to revenue.  
If popularity or number of votes are also important, this suggests that audience attention and visibility may also contribute to financial success.


## 9. Machine Learning Conclusion

This section applied supervised machine learning models to predict movie revenue.  
Linear Regression was used as a baseline model, while Decision Tree and Random Forest were used to capture nonlinear patterns.  
The model comparison shows which approach performs best according to MAE, RMSE, and R² score.  
Overall, this ML step extends the previous EDA and hypothesis testing by moving from relationship analysis to prediction.
